# Integration Layer Data Quality Validation

## Purpose

This notebook validates the relationships between the Silver datasets before creating the Integration layer.

The main goal is to identify unmatched records before joining the datasets so that no records are silently dropped.

### Relationships being validated

1. Green Taxi pickup location → Taxi Zone
2. Green Taxi dropoff location → Taxi Zone
3. Green Taxi pickup hour → Weather hour

The results will determine the appropriate join strategy for the Integration layer.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS `ftw-week-08`.`04-integration`;

CREATE TABLE IF NOT EXISTS `ftw-week-08`.`04-integration`.integration_dq_results (
    run_id STRING,
    executed_at TIMESTAMP,
    layer STRING,
    dataset STRING,
    check_name STRING,
    check_type STRING,
    status STRING,
    severity STRING,
    fail_count BIGINT,
    total_count BIGINT,
    fail_pct DOUBLE,
    threshold_pct DOUBLE,
    metric_value DOUBLE,
    owner STRING,
    details STRING
);

In [0]:
%sql
DECLARE OR REPLACE VARIABLE dq_run_id STRING;

SET VARIABLE dq_run_id = uuid();

SELECT
    dq_run_id AS run_id,
    current_timestamp() AS executed_at;

## 1. Pickup Location → Taxi Zone

### What are we checking?

Each Green Taxi trip contains a `pickup_location_id`.

We check whether this ID exists in the Taxi Zone Silver table.

### Expected result

We expect every pickup location ID to have a matching Taxi Zone record.

An unmatched record would mean that the taxi trip's pickup location cannot be linked to the zone dimension.

In [0]:
%sql
WITH check_result AS (
    SELECT
        COUNT(*) AS total_count,
        SUM(
            CASE
                WHEN z.location_id IS NULL THEN 1
                ELSE 0
            END
        ) AS unmatched_count
    FROM `ftw-week-08`.`03-silver`.green_taxi_clean t
    LEFT JOIN `ftw-week-08`.`03-silver`.taxi_zones_clean z
        ON t.pickup_location_id = z.location_id
)

INSERT INTO `ftw-week-08`.`04-integration`.integration_dq_results
SELECT
    dq_run_id,
    current_timestamp(),
    'integration',
    'green_taxi_to_taxi_zone',
    'pickup_location_zone_match',
    'referential_integrity',

    CASE
        WHEN unmatched_count = 0 THEN 'PASS'
        ELSE 'WARN'
    END AS status,

    CASE
        WHEN unmatched_count = 0 THEN 'PASS'
        ELSE 'WARN'
    END AS severity,

    unmatched_count AS fail_count,
    total_count,

    ROUND(
        unmatched_count * 100.0 / total_count,
        2
    ) AS fail_pct,

    0.0 AS threshold_pct,

    unmatched_count AS metric_value,

    'data_engineering' AS owner,

    CONCAT(
        'Pickup locations matched to Taxi Zones: ',
        CAST(total_count - unmatched_count AS STRING),
        '; unmatched: ',
        CAST(unmatched_count AS STRING),
        '; unmatched percentage: ',
        CAST(
            ROUND(unmatched_count * 100.0 / total_count, 2)
            AS STRING
        ),
        '%.'
    ) AS details

FROM check_result;

## 2. Dropoff Location → Taxi Zone

### What are we checking?

Each Green Taxi trip contains a `dropoff_location_id`.

We check whether this ID exists in the Taxi Zone Silver table.

### Expected result

We expect every dropoff location ID to have a matching Taxi Zone record.

In [0]:
%sql
WITH check_result AS (
    SELECT
        COUNT(*) AS total_count,
        SUM(
            CASE
                WHEN z.location_id IS NULL THEN 1
                ELSE 0
            END
        ) AS unmatched_count
    FROM `ftw-week-08`.`03-silver`.green_taxi_clean t
    LEFT JOIN `ftw-week-08`.`03-silver`.taxi_zones_clean z
        ON t.dropoff_location_id = z.location_id
)

INSERT INTO `ftw-week-08`.`04-integration`.integration_dq_results
SELECT
    dq_run_id,
    current_timestamp(),
    'integration',
    'green_taxi_to_taxi_zone',
    'dropoff_location_zone_match',
    'referential_integrity',

    CASE
        WHEN unmatched_count = 0 THEN 'PASS'
        ELSE 'WARN'
    END AS status,

    CASE
        WHEN unmatched_count = 0 THEN 'PASS'
        ELSE 'WARN'
    END AS severity,

    unmatched_count AS fail_count,
    total_count,

    ROUND(
        unmatched_count * 100.0 / total_count,
        2
    ) AS fail_pct,

    0.0 AS threshold_pct,

    unmatched_count AS metric_value,

    'data_engineering' AS owner,

    CONCAT(
        'Dropoff locations matched to Taxi Zones: ',
        CAST(total_count - unmatched_count AS STRING),
        '; unmatched: ',
        CAST(unmatched_count AS STRING),
        '; unmatched percentage: ',
        CAST(
            ROUND(unmatched_count * 100.0 / total_count, 2)
            AS STRING
        ),
        '%.'
    ) AS details

FROM check_result;

## 3. Trip → Weather Hour

### What are we checking?

Each Green Taxi trip is matched to weather data using the trip's pickup hour.

We compare:

- Green Taxi `pickup_datetime_local`
- Weather `observation_timestamp_local`

Both are truncated to the hour before matching.

### Expected result

Ideally, every taxi trip would have a corresponding weather observation.

However, the two datasets have different coverage periods, so some unmatched records may be expected.

We will investigate and quantify any unmatched records instead of automatically treating them as bad data.

In [0]:
%sql
WITH taxi_hours AS (
    SELECT
        date_trunc('hour', pickup_datetime_local) AS pickup_hour_local
    FROM `ftw-week-08`.`03-silver`.green_taxi_clean
    WHERE pickup_datetime_local IS NOT NULL
),

weather_hours AS (
    SELECT DISTINCT
        date_trunc('hour', observation_timestamp_local) AS weather_hour_local
    FROM `ftw-week-08`.`03-silver`.weather_hourly
),

check_result AS (
    SELECT
        COUNT(*) AS total_count,
        SUM(
            CASE
                WHEN w.weather_hour_local IS NULL THEN 1
                ELSE 0
            END
        ) AS unmatched_count
    FROM taxi_hours t
    LEFT JOIN weather_hours w
        ON t.pickup_hour_local = w.weather_hour_local
)

INSERT INTO `ftw-week-08`.`04-integration`.integration_dq_results
SELECT
    dq_run_id,
    current_timestamp(),
    'integration',
    'green_taxi_to_weather',
    'trip_weather_hour_match',
    'referential_integrity',

    CASE
        WHEN unmatched_count = 0 THEN 'PASS'
        ELSE 'WARN'
    END AS status,

    'WARN' AS severity,

    unmatched_count AS fail_count,
    total_count,

    ROUND(
        unmatched_count * 100.0 / total_count,
        2
    ) AS fail_pct,

    0.0 AS threshold_pct,

    unmatched_count AS metric_value,

    'data_engineering' AS owner,

    CONCAT(
        'Weather hour matches: ',
        CAST(total_count - unmatched_count AS STRING),
        '; unmatched: ',
        CAST(unmatched_count AS STRING),
        '; unmatched percentage: ',
        CAST(
            ROUND(unmatched_count * 100.0 / total_count, 2)
            AS STRING
        ),
        '%. Known coverage gaps: 175 trips occur on May 31, 2026 from 20:00-23:00 local time after available weather coverage ends; 5 trips occur before the March 1, 2026 weather coverage period. Trips should be retained using LEFT JOIN with explicit weather match status.'
    ) AS details

FROM check_result;

In [0]:
%sql
SELECT
    check_name,
    status,
    severity,
    fail_count,
    total_count,
    fail_pct,
    metric_value,
    details
FROM `ftw-week-08`.`04-integration`.integration_dq_results
WHERE run_id = dq_run_id
ORDER BY check_name;

## Analysis of Integration DQ Results

### Summary

| Relationship | Total Records | Matched | Unmatched | Unmatched % |
|---|---:|---:|---:|---:|
| Pickup → Taxi Zone | 133,353 | 133,353 | 0 | 0.00% |
| Dropoff → Taxi Zone | 133,353 | 133,353 | 0 | 0.00% |
| Trip → Weather Hour | 133,353 | 133,173 | 180 | 0.13% |

### Findings

#### Taxi Zone relationships

All pickup and dropoff location IDs matched the Taxi Zone dimension.

There were no unmatched pickup or dropoff locations, so the Taxi Zone relationships do not create a record-loss issue.

#### Weather relationship

180 taxi trips did not have a matching weather hour.

We investigated these records and found that they are explained by the available weather data coverage:

- 175 trips occurred on May 31, 2026 between 20:00–23:00 local time, after the available weather coverage ends.
- 5 trips occurred before March 1, 2026, which is before the available weather coverage period.

Therefore, the 180 unmatched records are a coverage issue rather than an unmatched Taxi Zone issue.

### Integration Decision

We will use `LEFT JOIN` from the Green Taxi dataset when creating the Integration layer.

This ensures that all valid taxi trips are retained even when weather data is unavailable.

For trips without a matching weather observation:

- The taxi trip will remain in the Integration dataset.
- Weather fields will be `NULL`.
- `weather_match_status` will identify that no weather record was found.

The same approach will be used for the Taxi Zone relationships with explicit match-status fields.